# ETL – DANE Exports (National Administrative Department of Statistics of Colombia)

This notebook implements an ETL pipeline to **download**, **extract**, **convert**, **consolidate**, and **clean** Colombia’s goods export data, covering the period **January 2023 – October 2025**.

The process design prioritizes computational efficiency, traceability, and scalability, using DuckDB as the main transformation engine and the Parquet format as the foundation of the analytical data lake.

The processing flow follows this structure:

**Monthly ZIP/XLSX files → XLSX to monthly Parquet conversion (bronze) → SQL transformations in DuckDB → Final enriched Parquet (gold).**

## Context

Official statistics on goods exports in Colombia are produced by DANE, based on administrative records from Export Declarations (DEX) submitted to DIAN through the MUISCA system.

Each DEX represents the basic unit of observation. Once a declaration is closed, the information is validated, cleaned, and consolidated by DANE for the production of official foreign trade statistics.

Although DANE publishes results on a monthly basis, historical information is distributed across independent ZIP files by period, which contain Excel spreadsheets, and there is no:

- **public API**,
- **aggregated and structured dataset** for longitudinal analysis that facilitates historical exploration of export operations at the micro (transaction) or meso (product, destination country, department, customs office, or modality) level.

As a result, **covering the period between January 2023 and October 2025 would require performing 58 individual downloads**, extracting each file and manually consolidating the information. This process is highly demanding in operational and computational terms, especially due to the use of Excel files, which require large amounts of RAM for processing.

## Project objective

The main objective of this project is to **design and document a reproducible process for consolidating and transforming export data**, enabling:

- Integration of the monthly reports published by DANE into a **single, coherent historical dataset**.
- Application of **data cleaning, standardization, and quality control rules** to key variables (dates, FOB values, weights, variable typing, and tariff classifications).
- Generation of a **structured analytical data source** that can serve as input for:
  - visualization and business intelligence exercises,
  - foreign trade and export performance analysis,
  - future training of statistical or machine learning models.

This notebook **does not seek to replace or reinterpret** DANE’s official methodology, but rather to **facilitate analytical access** to the information, while respecting the definitions, classifications, and scope established in the technical documentation of the export statistics operation.

## Reference documentation

For methodological details, variable definitions, classifications, validation rules, and statistical scope, it is recommended to consult DANE’s official documentation:

- *Colombia – Export Statistics (EXPO)*  
  Methodology and Statistical Production Directorate (DIMPE), DANE:  
  https://www.dane.gov.co/index.php/estadisticas-por-tema-2/comercio-internacional/exportaciones-1

This document describes, among other aspects:
- the structure and content of export declarations (DEX),
- classification and analysis variables (destination country, tariff subheading, FOB value, weights, modality, etc.),
- validation, consolidation, and quality control processes applied by DANE,
- and the conceptual framework under which official foreign trade statistics are produced.


## Repo structure

```
data-analytics-portfolio-nicolas-yepes/
├── README.md                          # Portfolio landing README
├── projects/
│   └── dian-export-etl/
│       ├── README.md                  # Project-specific ETL README
│       ├── notebooks/
│       │   └── ETL_Master_file_github.ipynb
│       ├── src/
│       │   ├── dian_export_sync.py        # Incremental ZIP download from DIAN/DANE
│       │   ├── dian_extract_zip.py        # ZIP extraction to XLSX
│       │   ├── xlsx_to_parquet_duckdb.py  # Direct XLSX → monthly Parquet conversion (bronze)
│       │   ├── schema_exportaciones.py    # Cast definitions and base schema (BRONZE_CASTS)
│       │   ├── sql_builders.py             # SQL generators (dynamic CAST SELECT)
│       │   └── build_razon_social_map.py   # Incremental construction of exporter name mapping
│       ├── data/                        # Git-ignored (local data lake)
│       │   ├── raw_zip/                 # ZIP files downloaded from DIAN/DANE
│       │   ├── extracted/               # Extracted XLSX files by period
│       │   ├── parquet_raw/             # Monthly Parquet files (bronze)
│       │   ├── processed/               # Final consolidated Parquet (gold)
│       │   ├── catalogs/                # Auxiliary catalogs
│       │   │   ├── hs2_capitulos.csv    # Official HS2 catalog
│       │   │   └── razon_social_map.parquet # Normalized exporter name mapping
│       │   └── duckdb_tmp/               # Temporary directory for disk spill (DuckDB)
│       ├── logs/                        # Git-ignored (execution logs)
│       │   └── .gitkeep
│       ├── requirements.txt             # Project dependencies
│   ## How to run

1. Create a virtual environment and install dependencies:
   - `pip install -r requirements.txt`
2. Ensure all custom project modules are located in the `src/` directory and that it is available in the Python path.
3. Run the notebook cells sequentially, in the order they are defined.
cutar el notebook en orden.


In [1]:
from pathlib import Path
from importlib import reload
import sys
import pandas as pd

# Current working directory
HERE = Path.cwd().resolve()

# Project root detection
if (HERE / "src").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "src").exists():
    REPO_ROOT = HERE.parent
else:
    raise RuntimeError("Project root could not be detected.")

# Directory definitions
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_zip"
EXTRACTED_DIR = DATA_DIR / "extracted"

# NEW: Monthly Parquet output (direct XLSX → Parquet using DuckDB)
PARQUET_RAW_DIR = DATA_DIR / "parquet_raw"       # “bronze”: one Parquet per monthly file
PROCESSED_DIR = DATA_DIR / "processed"           # “gold”: final clean/processed Parquet
LOG_DIR = REPO_ROOT / "logs"

# Create required directories
for d in [RAW_DIR, EXTRACTED_DIR, PARQUET_RAW_DIR, PROCESSED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Outputs (adjust names as needed)
OUTPUT_PARQUET_RAW_ALL = PARQUET_RAW_DIR / "exportaciones_raw_2023_2025.parquet"  # optional: consolidated raw
OUTPUT_PARQUET_FINAL = PROCESSED_DIR / "exportaciones_final.parquet"              # final clean output

# Add src/ to Python path
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Pandas configuration (still useful for inspection, although ETL is handled by DuckDB)
pd.set_option("display.max_columns", 200)




## 1) ZIP download (RAW)

This step downloads the ZIP files to the `data/raw_zip/` directory.  
If the files are already downloaded, this section can be skipped.


In [2]:
# Dynamic import of the DIAN/DANE download module
# This block dynamically imports and reloads the module
# responsible for synchronizing monthly export ZIP files.

try:
    # Import the full module
    import dian_export_sync

    # Force reload to reflect recent changes in "dian_export_sync.py"
    reload(dian_export_sync)

    # Import the main synchronization function that automatically
    # downloads ZIP files directly from the DIAN website
    from dian_export_sync import sync_dian_exportaciones

except Exception as e:
    # Controlled error with a clear message if the module is not available
    raise ImportError(
        "Unable to import 'dian_export_sync'. "
        "Please check that the file exists at ./src/dian_export_sync.py "
        "and that the 'src/' directory is correctly configured in the Python path."
    ) from e

# Incremental download of export ZIP files
# The function identifies and downloads only files
# that do not already exist locally, starting from
# the specified period (year and month).

new_files = sync_dian_exportaciones(
    dest_dir=RAW_DIR,    # Directory where raw ZIP files are stored
    desde_anio=2023,     # Initial year of the historical period
    desde_mes=1,         # Initial month
    verbose=True         # Detailed download logs
)

# Summary of files downloaded in this execution

print("New ZIP files downloaded:")
for f in new_files:
    # Print only the file name (not the full path)
    print(" -", Path(f).name if not isinstance(f, Path) else f.name)




Synchronizing exports 2023-01 → 2026-01

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\01_Exportaciones_2023_Enero.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\02_Exportaciones_2023_Febrero.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\03_Exportaciones_2023_Marzo.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\04_Exportaciones_2023_Abril.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\05_Exportaciones_2023_Mayo.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\06_Exportaciones_2023_Junio.zip

Saved to: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\raw_zip\07_Exportaciones_2023_Julio.zip

Saved to: E:\Portafo

## 2) Extracción de ZIPs a carpeta `extracted/`

Extrae los ZIPs descargados para dejar los XLSX/archivos de trabajo listos para conversión.


In [3]:
# Dynamic import of the ZIP extraction module
# This block imports and reloads the module responsible for
# extracting previously downloaded ZIP files.
# Using reload() allows reflecting code changes
# without restarting the kernel.

from importlib import reload

try:
    # Import the full module
    import dian_extract_zip

    # Force reload of the module (avoids using outdated src modules)
    reload(dian_extract_zip)

    # Import the main extraction function
    from dian_extract_zip import extract_all_zips

except Exception as e:
    # Controlled error with a clear message if the module is not available
    raise ImportError(
        "Unable to import 'dian_extract_zip'. "
        "Please check that the file exists at ./src/dian_extract_zip.py "
        "and that the 'src/' directory is correctly configured in the Python path."
    ) from e

# Incremental extraction of ZIP files
# Iterates over downloaded ZIPs and extracts their contents
# (mainly XLSX files), avoiding reprocessing files that already exist.

new_folders = extract_all_zips(
    zip_dir=str(RAW_DIR),            # Directory containing downloaded raw ZIP files
    extract_dir=str(EXTRACTED_DIR),  # Destination directory for extracted files
    overwrite=False,                 # Do not overwrite already extracted files
    verbose=True                     # Detailed process logging
)

# Extraction summary

print(f"ZIPs processed: {len(new_folders) if new_folders is not None else 'N/A'}")



 Extracting: 01_Exportaciones_2023_Enero.zip
   → Target folder: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2023_Enero
   Extracted successfully

 Extracting: 01_Exportaciones_2024_Enero.zip
   → Target folder: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2024_Enero
   Extracted successfully

 Extracting: 01_Exportaciones_2025_Enero.zip
   → Target folder: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\01_Exportaciones_2025_Enero
   Extracted successfully

 Extracting: 02_Exportaciones_2023_Febrero.zip
   → Target folder: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted\02_Exportaciones_2023_Febrero
   Extracted successfully

 Extracting: 02_Exportaciones_2024_Febrero.zip
   → Target folder: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-et

## 3) XLSX to Parquet conversion


In [8]:
import importlib
import xlsx_to_parquet_duckdb

# Reload the module to ensure the latest version is used
importlib.reload(xlsx_to_parquet_duckdb)
from xlsx_to_parquet_duckdb import xlsx_dir_to_parquet_dir


result = xlsx_dir_to_parquet_dir(
    xlsx_dir=EXTRACTED_DIR,
    out_dir=PARQUET_RAW_DIR,   # Output directory for monthly Parquet files (bronze layer)
    overwrite=False,  # If False, existing Parquet files will not be regenerated
    sheet=None,    # If None, the function will attempt to auto-detect the relevant sheet
    threads=4,
    memory_limit="12GB",  # Maximum memory allowed for DuckDB during conversion # Prevents out-of-memory errors when processing large Excel files
    use_parent_name=True, # Uses the parent folder name as the Parquet file name# Ensures traceability between source XLSX and output Parquet
    verbose=True,  # Enables detailed logging of the conversion process
    all_varchar=True,
    ignore_errors=False,
)

print(result)





[xlsx_to_parquet] Valid XLSX files found: 35 in E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\extracted
OK -> 01_Exportaciones_2023_Enero.parquet
OK -> 01_Exportaciones_2024_Enero.parquet
OK -> 01_Exportaciones_2025_Enero.parquet
OK -> 02_Exportaciones_2023_Febrero.parquet
OK -> 02_Exportaciones_2024_Febrero.parquet
OK -> 02_Exportaciones_2025_Febrero.parquet
OK -> 03_Exportaciones_2023_Marzo.parquet
OK -> 03_Exportaciones_2024_Marzo.parquet
OK -> 03_Exportaciones_2025_Marzo.parquet
OK -> 04_Exportaciones_2023_Abril.parquet
OK -> 04_Exportaciones_2024_Abril.parquet
OK -> 04_Exportaciones_2025_Abril.parquet
OK -> 05_Exportaciones_2023_Mayo.parquet
OK -> 05_Exportaciones_2024_Mayo.parquet
OK -> 05_Exportaciones_2025_Mayo.parquet
OK -> 06_Exportaciones_2023_Junio.parquet
OK -> 06_Exportaciones_2024_Junio.parquet
OK -> 06_Exportaciones_2025_Junio.parquet
OK -> 07_Exportaciones_2023_Julio.parquet
OK -> 07_Exportaciones_2024_Julio.parquet
OK -> 07_Exporta

### Exporter name mapping for normalization


In [19]:
from build_razon_social_map import build_razon_social_map

CATALOGS_DIR = DATA_DIR / "catalogs"
RS_MAP_PATH = CATALOGS_DIR / "razon_social_map.parquet"
# Notes: Path where the normalized exporter name mapping is stored
# This mapping is reused during the gold layer construction

build_razon_social_map(
    parquet_raw_dir=PARQUET_RAW_DIR,
    # Notes: Source directory containing monthly bronze Parquet files
    out_map=RS_MAP_PATH,
    col="RAZON_SOCIAL_EXPORTADOR",
    incremental=True,
    # Notes: Only new exporter names are processed, preserving existing mappings
)


Building full mapping: 14,616 unique values
Mapping created: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\catalogs\razon_social_map.parquet


WindowsPath('E:/Portafolio/data-analytics-portfolio-nicolas-yepes/projects/dian-export-etl/data/catalogs/razon_social_map.parquet')

### Creation of Parquet with the description of HS2 tariff chapters

In [54]:
from pathlib import Path
import re
import pandas as pd

# -------------------------------------------------------------------
# Input / output paths
# -------------------------------------------------------------------
HS2_TXT = (DATA_DIR / "catalogs" / "hs2_capitulos.txt").resolve()
HS2_PARQUET = (DATA_DIR / "catalogs" / "hs2_capitulos.parquet").resolve()

# -------------------------------------------------------------------
# Detect file encoding (UTF-16 with BOM vs latin-1 fallback)
# -------------------------------------------------------------------
raw = HS2_TXT.read_bytes()

if raw.startswith(b"\xff\xfe") or raw.startswith(b"\xfe\xff"):
    text = raw.decode("utf-16", errors="replace")
else:
    text = raw.decode("latin-1", errors="replace")

# -------------------------------------------------------------------
# Read non-empty lines
# -------------------------------------------------------------------
lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

rows = []

for ln in lines:
    # Normalize duplicated quotes
    ln = ln.replace('""', '"')

    # Skip header row (may include BOM or strange spaces)
    if "HS2" in ln.upper() and "DESCRIPCION" in ln.upper():
        continue

    # Skip lines without expected separator
    if ";" not in ln:
        continue

    left, right = ln.split(";", 1)

    # Extract numeric HS2 code
    hs2 = re.sub(r"[^0-9]", "", left)
    if not hs2:
        continue

    # Ensure HS2 is always two digits
    hs2 = hs2.zfill(2)

    # Clean description field
    desc = right.strip()

    # Remove repeated outer quotes
    desc = desc.strip().strip('"')

    # Normalize non-breaking spaces
    desc = desc.replace("\xa0", " ").strip()

    # Remove trailing duplicated quotes (e.g. miel natural"""")
    desc = re.sub(r'"{2,}$', "", desc).strip()

    rows.append((hs2, desc))

# -------------------------------------------------------------------
# Build DataFrame
# -------------------------------------------------------------------
df = pd.DataFrame(rows, columns=["HS2", "DESCRIPCION_HS2"])

# -------------------------------------------------------------------
# Final cleaning and validation
# -------------------------------------------------------------------
# Keep only valid 2-digit HS2 codes
df = df[df["HS2"].str.fullmatch(r"\d{2}", na=False)]

# Drop empty descriptions
df = df[df["DESCRIPCION_HS2"].astype(str).str.len() > 0]

# Remove duplicates and sort
df = df.drop_duplicates(subset=["HS2"]).sort_values("HS2")

# -------------------------------------------------------------------
# Basic checks
# -------------------------------------------------------------------
print("HS2 rows:", len(df))
print(df.head(10))

# -------------------------------------------------------------------
# Save to Parquet
# -------------------------------------------------------------------
df.to_parquet(HS2_PARQUET, index=False)
print("HS2 Parquet created:", HS2_PARQUET)


HS2 rows: 99
  HS2                                    DESCRIPCION_HS2
0  01                                     Animales vivos
1  02                      Carnes y despojos comestibles
2  03  Pescados y crustáceos\t moluscos e invertebrad...
3  04    Leche y productos lácteos; huevos; miel natural
4  05               Los demás productos de origen animal
5  06       Plantas vivas y productos de la floricultura
6  07  Hortalizas\t plantas\t raíces y tubérculos ali...
7  08  Frutas y frutos comestibles; cortezas de agrio...
8  09                  Café\t té\t yerba mate y especias
9  10                                           Cereales
HS2 Parquet created: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\catalogs\hs2_capitulos.parquet


### Creación de parquet final - Data lake gold

In [52]:
import duckdb
import importlib
import pandas as pd
import schema_exportaciones, sql_builders

# Reload modules to ensure latest definitions are used
importlib.reload(schema_exportaciones)
importlib.reload(sql_builders)

from schema_exportaciones import BRONZE_CASTS
from sql_builders import build_cast_select


# ============================================================
# 0) Config / Paths
# ============================================================
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PARQUET = OUTPUT_PARQUET_FINAL
raw_glob = (PARQUET_RAW_DIR / "*.parquet").as_posix()

# Canonical HS2 catalog (Parquet)
HS2_PARQUET = (DATA_DIR / "catalogs" / "hs2_capitulos.parquet").resolve()

# Optional: keep reference to original CSV just for logging (not used by DuckDB)
HS2_CSV = (DATA_DIR / "catalogs" / "hs2_capitulos.csv").resolve()

DUCKDB_TMP = (DATA_DIR / "duckdb_tmp").resolve()
DUCKDB_TMP.mkdir(parents=True, exist_ok=True)


# ============================================================
# 0.5) Ensure HS2 Parquet exists (no CSV parsing in DuckDB)
# ============================================================
if not HS2_PARQUET.exists():
    raise FileNotFoundError(
        f"No existe el catálogo HS2 en Parquet: {HS2_PARQUET}\n"
        f"Primero crea este archivo (desde tu hs2_capitulos_clean.csv o similar) "
        f"y vuelve a correr."
    )
else:
    print("HS2 catalog parquet OK:", HS2_PARQUET)


# ============================================================
# 1) Limit CASTs ONLY to final columns (reduces RAM usage)
# ============================================================
FINAL_COLS = {
    "ADUANA_SALIDA",
    "CANTIDAD_UNIDADES_FISICAS",
    "CIUDAD_DESTINATARIO",
    "CLASE_EXPORTADOR",
    "COD_LUG_SALIDA_ALF",
    "COD_LUGAR_SALIDA_NUM",
    "COD_MODALIDAD_EXPORTACION",
    "COD_MONEDA_TRANSACCION",
    "COD_PAIS_DESTINO",
    "COD_PAIS_DESTINO_ALF",
    "COD_UNIDAD_FISICA_ALF",
    "EXPORTACION_EN_TRANSITO",
    "FECHA_DECLARACION_EXPORTACION",
    "MODALIDAD_EXPORTACION",
    "MODO_TRANSPORTE",
    "NIT_DECLARANTE",
    "NIT_EXPORTADOR",
    "NUM_SOLICITUD_AUTO_EMBARQUE",
    "NUMERO_FORMULARIO",
    "PAIS_DESTINO_FINAL",
    "PESO_BRUTO_KGS",
    "PESO_NETO_KGS",
    "RAZON_SOCIAL_DECLARANTE",
    "RAZON_SOCIAL_DESTINATARIO",
    "RAZON_SOCIAL_EXPORTADOR",
    "REGION_DE_ORIGEN",
    "SISTEMAS_ESPECIALES",
    "SUBPARTIDA",
    "TIPO_CERTIFICADO_ORIGEN",
    "TIPO_DE_EMBARQUE",
    "TIPO_DECLARACION",
    "TIPO_DESPACHO",
    "UNIDAD_FISICA",
    "VALOR_FOB_PESOS",
    "VALOR_FOB_USD",
    "VALOR_SERIE_FLETES_USD",
    "VALOR_SERIE_SEGUROS_USD",
    "VLR_SERIE_AGREGADO_NAL_USD",
    "VLR_SERIE_OTROS_GASTOS_USD",
}

BRONZE_CASTS_MIN = {k: v for k, v in BRONZE_CASTS.items() if k in FINAL_COLS}
cast_sql = build_cast_select(BRONZE_CASTS_MIN, src_alias="g")


# ============================================================
# 2) DuckDB connection + anti-OOM settings
# ============================================================
con = duckdb.connect()
con.execute("PRAGMA threads=4;")
con.execute("PRAGMA memory_limit='12GB';")
con.execute(f"PRAGMA temp_directory='{DUCKDB_TMP.as_posix()}';")
con.execute("PRAGMA preserve_insertion_order=false;")


# ============================================================
# 3) Temporary tables: exporter name map + HS2 catalog (Parquet)
# ============================================================
con.execute(f"""
CREATE OR REPLACE TEMP TABLE razon_social_map AS
SELECT *
FROM read_parquet('{RS_MAP_PATH.as_posix()}');
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE hs2_catalog AS
SELECT
  LPAD(REGEXP_REPLACE(CAST(HS2 AS VARCHAR), '[^0-9]', '', 'g'), 2, '0') AS HS2,
  CAST(DESCRIPCION_HS2 AS VARCHAR) AS DESCRIPCION_HS2
FROM read_parquet('{HS2_PARQUET.as_posix()}');
""")



# ============================================================
# 4) Final GOLD layer in a single COPY statement
# ============================================================
con.execute(f"""
COPY (
  SELECT
    {cast_sql},

    COALESCE(
      STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y-%m-%d'), '%Y%m%d'),
      STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y/%m/%d'), '%Y%m%d'),
      STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y%m%d'), '%Y%m%d'),
      STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%d/%m/%Y'), '%Y%m%d'),
      NULLIF(REGEXP_REPLACE(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '[^0-9]', '', 'g'), '')
    ) AS FECHA_DECL_YYYYMMDD,

    TRY_STRPTIME(
      COALESCE(
        STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y-%m-%d'), '%Y%m%d'),
        STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y/%m/%d'), '%Y%m%d'),
        STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%Y%m%d'), '%Y%m%d'),
        STRFTIME(TRY_STRPTIME(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '%d/%m/%Y'), '%Y%m%d'),
        NULLIF(REGEXP_REPLACE(CAST(g.FECHA_DECLARACION_EXPORTACION AS VARCHAR), '[^0-9]', '', 'g'), '')
      ),
      '%Y%m%d'
    )::DATE AS FECHA_DECLARACION_DATE,

    LPAD(
      REGEXP_REPLACE(CAST(g.SUBPARTIDA AS VARCHAR), '[^0-9]', '', 'g'),
      10, '0'
    ) AS SUBPARTIDA_10,

    SUBSTR(
      LPAD(REGEXP_REPLACE(CAST(g.SUBPARTIDA AS VARCHAR), '[^0-9]', '', 'g'), 10, '0'),
      1, 2
    ) AS HS2,

    COALESCE(r.limpia, CAST(g.RAZON_SOCIAL_EXPORTADOR AS VARCHAR)) AS RAZON_SOCIAL_EXPORTADOR_LIMPIA,

    h.DESCRIPCION_HS2 AS DESCRIPCION_HS2

  FROM read_parquet('{raw_glob}') g

  LEFT JOIN razon_social_map r
    ON CAST(g.RAZON_SOCIAL_EXPORTADOR AS VARCHAR) = r.original

  LEFT JOIN hs2_catalog h
    ON SUBSTR(
         LPAD(REGEXP_REPLACE(CAST(g.SUBPARTIDA AS VARCHAR), '[^0-9]', '', 'g'), 10, '0'),
         1, 2
       ) = h.HS2
)
TO '{FINAL_PARQUET.as_posix()}'
(FORMAT 'parquet', COMPRESSION 'snappy');
""")

con.close()

print("Final GOLD Parquet created at:", FINAL_PARQUET)
print("HS2 Parquet used:", HS2_PARQUET)
print("Exporter name map used:", RS_MAP_PATH)
print("DuckDB temp dir:", DUCKDB_TMP)


HS2 catalog parquet OK: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\catalogs\hs2_capitulos.parquet
Final GOLD Parquet created at: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\processed\exportaciones_final.parquet
HS2 Parquet used: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\catalogs\hs2_capitulos.parquet
Exporter name map used: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\catalogs\razon_social_map.parquet
DuckDB temp dir: E:\Portafolio\data-analytics-portfolio-nicolas-yepes\projects\dian-export-etl\data\duckdb_tmp


In [53]:
import duckdb

con = duckdb.connect()

df_top_hs2 = con.execute(f"""
SELECT
  HS2,
  DESCRIPCION_HS2,
  COUNT(*) AS total_registros
FROM read_parquet('{FINAL_PARQUET.as_posix()}')
WHERE DESCRIPCION_HS2 IS NOT NULL
GROUP BY HS2, DESCRIPCION_HS2
ORDER BY total_registros DESC
LIMIT 20;
""").fetchdf()

con.close()

df_top_hs2


,HS2,DESCRIPCION_HS2,total_registros
0,06,Plantas vivas y productos de la floricultura,670419
1,33,Aceites esenciales y resinoides; preparaciones...,318072
2,39,Plásticos y sus manufacturas,252920
3,62,Prendas y complementos de vestir\t excepto los...,230898
4,61,Prendas y complementos de vestir\t de punto,197321
5,17,Azúcares y artículos de confitería,141643
6,08,Frutas y frutos comestibles; cortezas de agrio...,137104
7,85,"Máquinas""\t aparatos y material eléctrico\t y ...",110106
8,84,"Reactores nucleares""\t calderas\t máquinas\t"" ...",103465
9,48,Papel y cartón; manufacturas de pasta de celul...,82832
